# IOAI — 2025 Contest Tricy Table Data (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_tables.csv'):
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-tricy-table-data', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train_tables.csv' if os.path.exists('data/train_tables.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Tricy Table Data · 모범답안

**핵심 통찰: "이상한 테이블" = 흩뿌려진 결측값.** 베이스라인은 결측을 모델에 맡겼지만(RMSE ≈ 6.2),
**결측을 적극적으로 복원**하면 크게 좋아집니다. 컬럼들이 서로 강하게 상관(특히 target 과 0.996 인 `feat_1`)
하므로 **반복 대치(IterativeImputer)** 로 비어 있는 값을 다른 컬럼들로 회귀-추정합니다(타깃 미사용 = 누수 없음).
대치본 모델 + 결측-네이티브 모델을 **앙상블**해 RMSE ≈ **4.8** 로 내립니다. 채점은 동일 held-out(index%5).

In [ ]:
import pandas as pd, numpy as np
from sklearn.experimental import enable_iterative_imputer   # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
rmse = lambda y, p: mean_squared_error(y, p) ** 0.5

tr = pd.read_csv('data/train_tables.csv')
FEATS = [f'feat_{i}' for i in range(9)] + ['day','hour','minute']
is_val = np.arange(len(tr)) % 5 == 0
a, b = tr[~is_val], tr[is_val]


In [ ]:
# (1) 반복 대치로 결측 복원 (특징만 사용 → 타깃 누수 없음)
imp = IterativeImputer(random_state=0, max_iter=10)
Xa = imp.fit_transform(a[FEATS]); Xb = imp.transform(b[FEATS])
m1 = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.08, random_state=0).fit(Xa, a['target'])
print('대치+HistGBM RMSE:', round(rmse(b['target'], m1.predict(Xb)), 4))

# (2) 결측-네이티브 모델과 앙상블 (서로 다른 결측 처리 → 상호보완)
m2 = HistGradientBoostingRegressor(max_iter=600, learning_rate=0.05, max_leaf_nodes=63,
                                   early_stopping=True, n_iter_no_change=20, random_state=1).fit(a[FEATS], a['target'])
pred = (m1.predict(Xb) + m2.predict(b[FEATS])) / 2
print('ENSEMBLE held-out RMSE:', round(rmse(b['target'], pred), 4))

pd.DataFrame({'id': range(len(b)), 'target': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(b))


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)